# Preprocessing Pipeline

This notebook covers the full preprocessing pipeline: extracting patches from raw SHG stacks, quality-checking them, normalizing intensities, and splitting into train/validation/test sets.

Raw `.tif` files were first exported from `.czi` using an ImageJ/Fiji macro (`extract_SHG_channels.ijm`) prior to this pipeline.

## 1. Extract Patches from Raw Stacks

Walks through raw backward/forward `.tif` file pairs and extracts non-overlapping 256×256 patches per z-slice, organized by stack name.

In [ ]:
import os
import numpy as np
import tifffile as tiff

PATCH_SIZE = 256
BASE_DIR = "/Users/nicolaslo/Downloads/SHG research"
DATA_DIR = os.path.join(BASE_DIR, "data", "tif files")
PATCH_ROOT = os.path.join(BASE_DIR, "data", "patches_by_stack")


def extract_patches(image_path, stack_name, channel_subdir):
    img = tiff.imread(image_path)

    if img.ndim == 2:
        img = img[np.newaxis, ...]

    z_slices, H, W = img.shape
    n_ph = H // PATCH_SIZE
    n_pw = W // PATCH_SIZE

    # e.g. patches_by_stack/F+B+Lambda_starting_from_500um/backward/
    out_root = os.path.join(PATCH_ROOT, stack_name, channel_subdir)
    os.makedirs(out_root, exist_ok=True)

    print(f"  [{stack_name}/{channel_subdir}] {z_slices} slices, {n_ph*n_pw} patches/slice")

    for z in range(z_slices):
        slice_img = img[z]
        for i in range(n_ph):
            for j in range(n_pw):
                patch = slice_img[
                    i*PATCH_SIZE:(i+1)*PATCH_SIZE,
                    j*PATCH_SIZE:(j+1)*PATCH_SIZE,
                ]
                out_name = f"z{z:03d}_r{i}_c{j}.tif"
                tiff.imwrite(os.path.join(out_root, out_name), patch)


def clean_stack_name(fname):
    """Turn a backward filename into a safe stack folder name."""
    name = fname.replace("_ch2_backwards.tif", "")
    name = name.strip().replace("/", "-")
    return name


def main():
    pairs_found = 0
    pairs_processed = 0

    for root, dirs, files in os.walk(DATA_DIR):
        for fname in files:
            if not fname.endswith("_ch2_backwards.tif"):
                continue

            pairs_found += 1
            backward_path = os.path.join(root, fname)
            forward_name = fname.replace("_ch2_backwards.tif", "_ch5_forwards.tif")
            forward_path = os.path.join(root, forward_name)

            if not os.path.exists(forward_path):
                print(f"SKIP (no forward match): {fname}")
                continue

            stack_name = clean_stack_name(fname)
            print(f"Processing stack: {stack_name}")

            extract_patches(backward_path, stack_name, "backward")
            extract_patches(forward_path, stack_name, "forward")
            pairs_processed += 1

    print(f"\nFound {pairs_found} backward files, processed {pairs_processed} matched pairs.")


    main()

## 2. Check Patch Quality Across All Stacks

Scans every stack/channel folder to flag empty or near-empty patches and report basic intensity statistics.

In [ ]:
import os
import numpy as np
import tifffile as tiff

BASE = "/Users/nicolaslo/Downloads/SHG research/data/patches_by_stack"
EMPTY_THRESHOLD = 5  # max pixel value below this = likely empty patch

def analyze_folder(folder):
    files = [f for f in os.listdir(folder) if f.endswith(".tif")]
    if not files:
        return None

    maxes = []
    means = []
    empty_count = 0

    for f in files:
        img = tiff.imread(os.path.join(folder, f))
        m = img.max()
        maxes.append(m)
        means.append(img.mean())
        if m <= EMPTY_THRESHOLD:
            empty_count += 1

    return {
        "total": len(files),
        "empty": empty_count,
        "pct_empty": 100 * empty_count / len(files),
        "avg_max": np.mean(maxes),
        "avg_mean": np.mean(means),
        "dtype": img.dtype,
    }

print(f"{'Stack':30} {'Channel':10} {'Total':>7} {'Empty':>7} {'%Empty':>8} {'AvgMax':>10} {'AvgMean':>10}")

for stack in sorted(os.listdir(BASE)):
    stack_path = os.path.join(BASE, stack)
    if not os.path.isdir(stack_path):
        continue
    for channel in ["backward", "forward"]:
        ch_path = os.path.join(stack_path, channel)
        if not os.path.isdir(ch_path):
            continue
        stats = analyze_folder(ch_path)
        if stats is None:
            continue
        print(f"{stack:30} {channel:10} {stats['total']:>7} {stats['empty']:>7} "
              f"{stats['pct_empty']:>7.1f}% {stats['avg_max']:>10.1f} {stats['avg_mean']:>10.2f}")

## 3. Flatten Nested Stack Folders

Stacks 5 and 6 were originally saved with an extra nested subfolder level; this step flattens them to match the structure of the other stacks.

In [ ]:
import os
import shutil

BASE = "/Users/nicolaslo/Downloads/SHG research/data/patches_by_stack"

groups_to_flatten = ["stack5_post_cornea", "stack6_cxl_posterior"]

for stack in groups_to_flatten:
    stack_path = os.path.join(BASE, stack)
    subfolders = [f for f in os.listdir(stack_path)
                  if os.path.isdir(os.path.join(stack_path, f))]

    for channel in ["backward", "forward"]:
        out_dir = os.path.join(stack_path, channel)
        os.makedirs(out_dir, exist_ok=True)

        for sub in subfolders:
            if sub in ("backward", "forward"):
                continue
            src = os.path.join(stack_path, sub, channel)
            if not os.path.isdir(src):
                continue
            for fname in os.listdir(src):
                new_name = f"{sub}__{fname}"
                shutil.copy(
                    os.path.join(src, fname),
                    os.path.join(out_dir, new_name)
                )
    print(f"Flattened: {stack}")

print("Done.")

## 4. Visual Sanity Check

Quick visual inspection of a single patch to confirm image content and intensity range look reasonable before normalization.

In [ ]:
import tifffile as tiff
import numpy as np
from PIL import Image
import os

folder = "/Users/nicolaslo/Downloads/SHG research/data/patches_by_stack/stack6_cxl_posterior/forward"
files = os.listdir(folder)
first_file = files[0]  # just grab the first one automatically

img = tiff.imread(os.path.join(folder, first_file))
print("Using file:", first_file)
print("min:", img.min(), "max:", img.max())

scaled = ((img - img.min()) / (img.max() - img.min()) * 255).astype(np.uint8)
Image.fromarray(scaled).save("/Users/nicolaslo/Downloads/SHG research/test_view.png")
print("Saved preview to test_view.png")

## 5. Percentile Normalization

Applies global 1st/99th percentile normalization per stack/channel to reduce the impact of intensity outliers, scaling all patches to the [0, 1] range.

In [ ]:
import os
import numpy as np
import tifffile as tiff

BASE = "/Users/nicolaslo/Downloads/SHG research/data/patches_by_stack"
NORM_ROOT = "/Users/nicolaslo/Downloads/SHG research/data/normalized"

def normalize_stack(stack, channel):
    src = os.path.join(BASE, stack, channel)
    out = os.path.join(NORM_ROOT, stack, channel)
    os.makedirs(out, exist_ok=True)

    files = [f for f in os.listdir(src) if f.endswith(".tif")]
    all_pixels = []
    for f in files:
        img = tiff.imread(os.path.join(src, f))
        all_pixels.append(img.flatten())
    all_pixels = np.concatenate(all_pixels)

    p1, p99 = np.percentile(all_pixels, [1, 99])
    print(f"{stack}/{channel}: p1={p1:.1f}, p99={p99:.1f}")

    for f in files:
        img = tiff.imread(os.path.join(src, f)).astype(np.float32)
        norm = np.clip((img - p1) / (p99 - p1), 0, 1)
        tiff.imwrite(os.path.join(out, f), norm.astype(np.float32))

stacks = [s for s in os.listdir(BASE) if os.path.isdir(os.path.join(BASE, s)) and not s.startswith(".")]
for stack in stacks:
    for channel in ["backward", "forward"]:
        normalize_stack(stack, channel)

print("Normalization complete.")

## 6. Train / Validation / Test Split

Splits normalized patches into train, validation, and test sets by stack, ensuring no data leakage between splits (each full stack is assigned entirely to one split).

In [ ]:
import os
import shutil

BASE = "/Users/nicolaslo/Downloads/SHG research/data/normalized"
SPLIT_ROOT = "/Users/nicolaslo/Downloads/SHG research/data/split"

split_assignment = {
    "stack1_500um_cornea10x": "train",
    "stack2_first100um_top": "train",
    "stack3_lenticule": "train",
    "stack5_post_cornea": "train",
    "stack4_small_area": "val",
    "stack6_cxl_posterior": "test",
}

for stack, split in split_assignment.items():
    for channel in ["backward", "forward"]:
        src = os.path.join(BASE, stack, channel)
        out_dir = os.path.join(SPLIT_ROOT, split, channel)
        os.makedirs(out_dir, exist_ok=True)

        for fname in os.listdir(src):
            new_name = f"{stack}__{fname}"
            shutil.copy(
                os.path.join(src, fname),
                os.path.join(out_dir, new_name)
            )
    print(f"Copied {stack} -> {split}")

print("Done.")

## 7. Final Sanity Check

Confirms the final split data loads correctly with expected dtype and intensity range.

In [ ]:
import tifffile as tiff
import os

folder = "/Users/nicolaslo/Downloads/SHG research/data/split/train/forward"
sample_file = os.listdir(folder)[0]
img = tiff.imread(os.path.join(folder, sample_file))

print("File:", sample_file)
print("dtype:", img.dtype)
print("min:", img.min(), "max:", img.max())